# River basin delineation
<br>
<img style="float: left; padding-right: 15px; padding-left: 0px;" src="../sources/images/logo_continuum.png" width="260px" align=”left” >

<div style="text-align: justify">
This notebook uses Continuum static data and GRASS GIS tools to delineate river sub-basins in the main basin modelled. The generated basin mask shapefile can be used to generate Continuum static data in sub-basins.


Where text mention **USER ACTION**, you are expected to update paths, names or flags
to match your specific project. Otherwise, you can usually run the cells in sequence
without modifying the code.


## 🔧 Preliminary Setup

This cell indicate the main basin in which the user is working and ask to specify a project names for the selected sub-basin.

**USER ACTION**: change the basin names here if you want to work on a different basin and specify the sub-basin name

In [47]:
# Settings
mainbasin_name = "awash"                           # choose among awash, danakil and shebele
mainbasin_project = mainbasin_name + "_iwrm"       # automatic

subbasin_name= "awashOmbole"                       # specify a name for the sub-basin
subbasin_project = subbasin_name + "_iwrm"         # automatic

#### Import libraries
Import libraries required by the script.

In [48]:
from pathlib import Path
from io import BytesIO
import base64
import os, shutil
import geopandas as gpd

import numpy as np
import rasterio
from rasterio.transform import rowcol, xy
from rasterio.features import shapes
from PIL import Image

from ipyleaflet import Map, basemaps, ImageOverlay, GeoJSON, Marker, Rectangle
from ipywidgets import Output

from grass_session import Session
from grass.pygrass.modules.shortcuts import raster as r, vector as v
from grass.pygrass.modules import Module
import grass.script as gs
from grass.script import parse_key_val

import hmc_tools

#### Set paths
Define input and output paths based on the user choices.

In [49]:
# Define the project root directory
projectroot = str(Path().cwd()).replace('notebook_tools','') + 'projects/'
basins_dict = {"mainbasin": mainbasin_name, "mainbasin_prj": mainbasin_project, "subbasin": subbasin_name, "subbasin_prj":  subbasin_project}

choice_mainbasin = (projectroot + "/{mainbasin_prj}/continuum/data_geo/gridded/{mainbasin}.choice.txt").format(**basins_dict)
pnt_mainbasin = (projectroot + "/{mainbasin_prj}/continuum/data_geo/gridded/{mainbasin}.pnt.txt").format(**basins_dict)

## 📍 Identify sub-basin outlet
This cell generate a map overlayed by the model river network. 

**USER ACTION**: the user should choose a location to be set as the closing section of the selected basin. The section should lie on the blue river network.

In [50]:
# Read domain river network
with rasterio.open(choice_mainbasin) as src:
    data = src.read(1)
    bounds = src.bounds         
    crs = src.crs
    transform = src.transform   
    print("CRS:", crs)
    print("Bounds:", bounds)

h, w = data.shape

# Vectorialise network
mask = data == 1

features = []
for geom, val in rasterio.features.shapes(data.astype(np.int16), mask=mask, transform=transform):
    features.append({
        "type": "Feature",
        "geometry": geom,
        "properties": {"value": int(val)}
    })

geojson_data = {
    "type": "FeatureCollection",
    "features": features
}

# Draw map
left, bottom, right, top = bounds
center = ((bottom + top) / 2, (left + right) / 2)

m = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=center,
    zoom=10
)

# Vector layer: cells with value 1
geo_layer = GeoJSON(
    data=geojson_data,
    style={
        "color": "blue",
        "weight": 1,
        "fillColor": "blue",
        "fillOpacity": 0.3
    }
)
m.add_layer(geo_layer)

# Structure to store selected cell
selected_cell = {
    "row": None, "col": None,
    "click_lat": None, "click_lon": None,
    "cell_lat": None, "cell_lon": None
}

# Vector with clicked coordinates [lon, lat]
coord = [None, None]

# Marker on selected cell center
marker = Marker(location=center, draggable=False)
marker.visible = False
m.add_layer(marker)

# Rectangle highlighting the exact raster cell
cell_rect = Rectangle(
    bounds=((center[0], center[1]), (center[0], center[1])),
    color="red",
    fill=False,
    weight=3
)
cell_rect.visible = False
m.add_layer(cell_rect)

out = Output()

def handle_interaction(**kwargs):
    if kwargs.get("type") == "click":
        lat, lon = kwargs.get("coordinates")  # (lat, lon) in WGS84
        
        # Update coord vector
        coord[:] = [lon, lat]
        
        selected_cell["click_lat"] = lat
        selected_cell["click_lon"] = lon
        
        # From WGS84 coordinates to raster row/col
        r, c = rowcol(transform, lon, lat)  # NOTE: lon, lat!
        selected_cell["row"] = int(r)
        selected_cell["col"] = int(c)
        
        if 0 <= r < h and 0 <= c < w:
            val = data[r, c]
            
            # Cell center
            x_center, y_center = xy(transform, r, c, offset="center")
            cell_lon, cell_lat = x_center, y_center
            selected_cell["cell_lat"] = cell_lat
            selected_cell["cell_lon"] = cell_lon
            
            # Marker on cell center
            marker.location = (cell_lat, cell_lon)
            marker.visible = True
            
            # Exact cell bounding box
            x_ul, y_ul = xy(transform, r, c, offset="ul")
            x_lr, y_lr = xy(transform, r, c, offset="lr")
            # Leaflet expects ((lat_min, lon_min), (lat_max, lon_max))
            cell_rect.bounds = ((y_lr, x_ul), (y_ul, x_lr))
            cell_rect.visible = True
            
            with out:
                out.clear_output()
                print("📍 Clicked coordinates:")
                print(f"   clicked coord = [{lon:.8f}, {lat:.8f}]")

                if val ==1:
                    print("✅ Selected raster cell has coordinates:")
                    print(f"   lat = {cell_lon:.8f}, lon = {cell_lat:.8f}")
                
                if val != 1:
                    print("⚠️ WARNING!: clicked cell is NOT on the network.")
        else:
            with out:
                out.clear_output()
                print("⚠️ WARNING: click is outside raster extent.")
                print(f"   clicked coord = [{lon:.8f}, {lat:.8f}]")

m.on_interaction(handle_interaction)

display(m)
display(out)


CRS: None
Bounds: BoundingBox(left=37.986155947778, bottom=7.897662325, right=43.328126311178, top=12.304338214)


Map(center=[10.1010002695, 40.657141129478006], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zo…

Output()

## 🏞️ Basin delineation

#### Start GRASS Session

This cell initializes a GRASS GIS session. It sets up the database, location, and mapset. If the mapset doesn't exist, it is created. This session provides the environment for all subsequent GRASS GIS commands.

In [51]:
gisdb = "/home/continuumuser/grassdata"
location = "WGS84"
mapset = f"{subbasin_name}_basin"   

location_path = os.path.join(gisdb, location)
perm_path     = os.path.join(location_path, "PERMANENT")
mapset_path   = os.path.join(location_path, mapset)
lockfile      = os.path.join(mapset_path, ".gislock")

# Create location if missing
if not os.path.isdir(perm_path):
    print(f"Creating GRASS LOCATION {location} (EPSG:4326)...")
    subprocess.run(["grass", "--text", "-c", "EPSG:4326", perm_path], check=True)
else:
    print(f"LOCATION {location} already exists.")

# Clean lock files
if os.path.exists(lockfile):
    try:
        os.remove(lockfile)
        print(f"Removed stale lock: {lockfile}")
    except Exception as e:
        print(f"Warning: cannot remove lock {lockfile}: {e}")

if os.path.isdir(mapset_path):
    print(f"Removing existing MAPSET: {mapset_path}")
    shutil.rmtree(mapset_path)

print(f"Creating new MAPSET: {mapset_path}")
os.makedirs(mapset_path, exist_ok=True)

# Coopy required files from PERMANENT
for fname in ("WIND", "DEFAULT_WIND", "PROJ_INFO", "PROJ_UNITS"):
    src = os.path.join(perm_path, fname)
    dst = os.path.join(mapset_path, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)

# Explicitly export GISRC
gisrc_path = f"/tmp/grassrc_{location}_{mapset}"
with open(gisrc_path, "w") as f:
    f.write(f"GISDBASE: {gisdb}\n")
    f.write(f"LOCATION_NAME: {location}\n")
    f.write(f"MAPSET: {mapset}\n")

os.environ["GISRC"] = gisrc_path
print("GISRC set to:", os.environ["GISRC"])
print("✅ GRASS GIS is ready!")
# Register uncovnentional pygrass lirbaries
r.in_gdal        = Module("r.in.gdal")
v.out_ogr       = Module("v.out.ogr")
r.to_vect       = Module("r.to.vect")
r.stream_basins  = Module("r.stream.basins")  

LOCATION WGS84 already exists.
Creating new MAPSET: /home/continuumuser/grassdata/WGS84/awashOmbole_basin
GISRC set to: /tmp/grassrc_WGS84_awashOmbole_basin
✅ GRASS GIS is ready!


#### Import macrobasin map into GRASS GIS and convert drainage direction from Continuum format
This cell imports raster data into GRASS GIS and convert from Continuum to GRASS format the drainage direction.

In [52]:
print("Importing Drainage Direction into GRASS GIS")
# Import the processed and conditioned DEMs
r.in_gdal(input=pnt_mainbasin, output="pnt", flags='o', overwrite=True, quiet=True)
gs.run_command("g.region", raster="pnt")

# Convert Continuum drainage direction to grass
conversion_file = "/home/continuumuser/cont2grass.txt"
hmc_tools.writeHMC2Grass(os.path.dirname(conversion_file))      # write a temporary file cont2grass.txt to convert to grass format
r.reclass(input="pnt", output="pnt_grass", rules=conversion_file, overwrite=True, quiet=True)
os.remove(conversion_file)
print("✅ All maps are ready!")

Importing Drainage Direction into GRASS GIS
✅ All maps are ready!


#### Sub basin extraction
This cell delinate the sub-basin and exports the GRASS mask as an ESRI Shapefile on disk in the project folder.

In [45]:
# Extract basin mask
r.stream_basins(direction="pnt_grass", coordinates= coord, basins="basin", overwrite=True, quiet=True)

mask_file = (projectroot + "/{subbasin_prj}/data/{subbasin}_mask.shp").format(**basins_dict)
os.makedirs(os.path.dirname(mask_file), exist_ok=True)

r.to_vect(overwrite=True, input="basin", output="basin_mask", type="area", quiet=True)

try:
    v.out_ogr(input="basin_mask", output=mask_file, format="ESRI_Shapefile", overwrite=True, quiet=True)
except:
    raise ValueError("Mask can not be written... Is the shapefile open in QGIS?")
    
print("✅ Mask saved to " + mask_file.replace('/home/continuumuser/workdir/', "") + " !")

✅ Mask saved to projects//awash-abovedam_iwrm/data/awash-abovedam_mask.shp !


In [46]:
gs.run_command("g.region", vector="basin_mask")
gs.run_command("g.region", flags="p")

projection: 3 (Latitude-Longitude)
zone:       0
datum:      wgs84
ellipsoid:  wgs84
north:      9:18:34.55011N
south:      8:18:08.485378N
west:       37:59:42.53699E
east:       39:04:59.981923E
nsres:      0:00:32.375578
ewres:      0:00:32.375578
rows:       112
cols:       121
cells:      13552


#### Plot sub-basin mask
Run this cell after the basin delineation has completed successfully to visually check the result.

In [26]:
basin_gdf = gpd.read_file(mask_file)

# Ensure CRS is WGS84 for correct overlay on OSM
if basin_gdf.crs is None:
    raise ValueError("Basin shapefile has no CRS defined. Please set it before using this cell.")
if basin_gdf.crs.to_epsg() != 4326:
    basin_gdf = basin_gdf.to_crs(epsg=4326)

basin_geojson = basin_gdf.__geo_interface__

# Compute map center from basin bounds
minx, miny, maxx, maxy = basin_gdf.total_bounds
center_basin = ((miny + maxy) / 2, (minx + maxx) / 2)

m_basin = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=center_basin,
    zoom=11
)

basin_layer = GeoJSON(
    data=basin_geojson,
    style={
        "color": "red",
        "weight": 2,
        "fillColor": "red",
        "fillOpacity": 0.2
    },
    name="Delineated basin"
)
m_basin.add_layer(basin_layer)

display(m_basin)

Map(center=[np.float64(8.4687315473356), np.float64(38.827021653117285)], controls=(ZoomControl(options=['posi…